# Paradigmas de Programação — Aula 1
## Notebook 2 de 3 — O mapa de paradigmas de Van Roy

**Universidade La Salle · Disciplina: Paradigmas de Programação**

**Objetivo:** reconstruir, em escala reduzida, o pôster *Programming Paradigms for Dummies* de Peter Van Roy. Cada paradigma é posicionado em dois eixos — **estado observável** e **modelo de concorrência** — e as linguagens do semestre são plotadas sobre esse mapa.

> Como usar: menu *Ambiente de execução → Executar tudo* (`Ctrl+F9`). Nada precisa ser instalado — o Colab já traz `numpy`, `pandas` e `matplotlib`.


In [ ]:
# Configuração comum a todos os notebooks da disciplina
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# Paleta da disciplina: um paradigma, uma cor
CORES = {
    "Imperativo": "#B85042",
    "Funcional":  "#2C5F2D",
    "Orientado a objetos": "#065A82",
    "Lógico":     "#6D2E46",
    "Concorrente": "#C77D00",
    "Declarativo": "#50808E",
}
print("Ambiente pronto. matplotlib", matplotlib.__version__, "| numpy", np.__version__)


---
## 1. A ideia da taxonomia

Van Roy organiza os paradigmas por **acréscimo de conceitos** a um núcleo declarativo. Dois eixos
dão conta da maior parte das distinções que nos interessam neste semestre:

| Eixo | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| **Estado observável** | nenhum (puro) | encapsulado / controlado | explícito local | explícito global e compartilhado |
| **Concorrência** | nenhuma | declarativa (determinística) | por mensagens (isolada) | por estado compartilhado |

O ponto pedagógico: **cada conceito acrescentado ganha expressividade e perde uma garantia**.


In [ ]:
import pandas as pd

paradigmas = pd.DataFrame([
    # nome,                        estado, concorrência, família,     linguagem-exemplo
    ("Funcional puro",                  0, 0, "Funcional",           "Haskell"),
    ("Funcional com efeitos",           1, 0, "Funcional",           "Elixir, OCaml"),
    ("Lógico puro",                     0, 0, "Lógico",              "Prolog puro"),
    ("Lógico com restrições",           0, 1, "Lógico",              "CLP(FD), ASP"),
    ("Orientado a objetos",             2, 0, "Orientado a objetos", "Java, Python"),
    ("Imperativo procedural",           3, 0, "Imperativo",          "C, Pascal"),
    ("Concorrente declarativo",         0, 1, "Funcional",           "dataflow"),
    ("Atores / troca de mensagens",     1, 2, "Concorrente",         "Elixir, Erlang"),
    ("CSP / canais",                    1, 2, "Concorrente",         "Go"),
    ("Threads com memória compartilhada", 3, 3, "Concorrente",       "Java, C++"),
    ("Orientado a arrays",              0, 1, "Declarativo",         "NumPy, APL"),
    ("Reativo",                         1, 1, "Declarativo",         "RxJS"),
], columns=["paradigma", "estado", "concorrencia", "familia", "exemplos"])

paradigmas


---
## 2. O mapa

Um pequeno deslocamento aleatório (*jitter*) separa paradigmas que caem na mesma coordenada —
o mapa é conceitual, não métrico.


In [ ]:
rng = np.random.default_rng(42)
jx = rng.uniform(-0.13, 0.13, len(paradigmas))
jy = rng.uniform(-0.13, 0.13, len(paradigmas))

fig, ax = plt.subplots(figsize=(11, 7))

for familia, grupo in paradigmas.groupby("familia"):
    idx = grupo.index
    ax.scatter(grupo["estado"] + jx[idx], grupo["concorrencia"] + jy[idx],
               s=260, alpha=0.85, label=familia,
               color=CORES.get(familia, "#888888"), edgecolor="white", linewidth=1.5, zorder=3)

for i, linha in paradigmas.iterrows():
    ax.annotate(f"{linha['paradigma']}\n({linha['exemplos']})",
                (linha["estado"] + jx[i], linha["concorrencia"] + jy[i]),
                textcoords="offset points", xytext=(0, 16),
                ha="center", fontsize=8, color="#333333")

ax.set_xticks(range(4))
ax.set_xticklabels(["nenhum\n(puro)", "encapsulado", "explícito\nlocal", "explícito\ncompartilhado"])
ax.set_yticks(range(4))
ax.set_yticklabels(["sem\nconcorrência", "declarativa", "por\nmensagens", "estado\ncompartilhado"])
ax.set_xlabel("Estado observável  →  mais poder, menos garantias", weight="bold")
ax.set_ylabel("Modelo de concorrência", weight="bold")
ax.set_xlim(-0.5, 3.6); ax.set_ylim(-0.5, 3.6)
ax.set_title("Mapa de paradigmas segundo os eixos de Van Roy", fontsize=14, weight="bold")
ax.legend(title="Família", loc="upper left", frameon=True)

# a diagonal do risco: quanto mais para cima e para a direita, mais difícil de raciocinar
ax.plot([-0.5, 3.6], [-0.5, 3.6], ls="--", color="#999999", lw=1, zorder=1)
ax.annotate("mais difícil de\nraciocinar e testar", (3.3, 3.35), ha="center",
            fontsize=9, style="italic", color="#777777")

fig.tight_layout()
plt.show()


### Como ler o mapa

- O **canto inferior esquerdo** é o núcleo declarativo: sem estado observável e sem concorrência.
  Tudo que está lá tem transparência referencial — o mesmo programa, a mesma resposta, sempre.
- Cada passo **para a direita** acrescenta estado; cada passo **para cima** acrescenta
  não determinismo. Os dois juntos, no canto superior direito, produzem o cenário mais caro de
  depurar: *threads* mutando memória compartilhada.
- Elixir e Go aparecem duas vezes, em posições diferentes. Isso não é erro: **linguagens
  multiparadigma ocupam regiões, não pontos**. O paradigma é uma propriedade do *programa que você
  escreve*, não do logotipo da linguagem.


---
## 3. Onde ficam as linguagens do semestre

In [ ]:
linguagens = pd.DataFrame([
    ("Python",  2.2, 2.4, "usada em quase todos os encontros"),
    ("Haskell", 0.2, 0.8, "funcional puro, Aulas 4 e 5"),
    ("Racket",  0.9, 0.4, "interpretadores e macros, Aulas 6 e 16"),
    ("Java",    2.8, 3.0, "orientação a objetos e threads, Aulas 11 e 14"),
    ("Prolog",  0.3, 0.2, "paradigma lógico, Aulas 12 e 13"),
    ("Elixir",  1.0, 2.0, "funcional e atores, Aulas 5 e 15"),
    ("Go",      2.4, 2.0, "CSP e canais, Aula 15"),
], columns=["linguagem", "estado", "concorrencia", "onde"])

fig, ax = plt.subplots(figsize=(10.5, 6.5))
ax.scatter(linguagens["estado"], linguagens["concorrencia"], s=420,
           color="#1E2761", alpha=0.9, edgecolor="white", linewidth=2, zorder=3)

for _, l in linguagens.iterrows():
    ax.annotate(l["linguagem"], (l["estado"], l["concorrencia"]),
                ha="center", va="center", color="white", fontsize=9, weight="bold", zorder=4)
    ax.annotate(l["onde"], (l["estado"], l["concorrencia"]),
                textcoords="offset points", xytext=(0, -26),
                ha="center", fontsize=8, color="#555555", style="italic")

ax.set_xticks(range(4))
ax.set_xticklabels(["nenhum", "encapsulado", "explícito", "compartilhado"])
ax.set_yticks(range(4))
ax.set_yticklabels(["nenhuma", "declarativa", "mensagens", "compartilhado"])
ax.set_xlabel("Estado observável que a linguagem incentiva", weight="bold")
ax.set_ylabel("Concorrência que a linguagem oferece", weight="bold")
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.5, 3.5)
ax.set_title("As sete linguagens do semestre no mesmo mapa", fontsize=14, weight="bold")
fig.tight_layout()
plt.show()


---
## 4. Exercícios

1. Acrescente três linguagens que você já usou (por exemplo Rust, SQL, JavaScript, C) à tabela
   `linguagens` e justifique **em uma frase** cada coordenada escolhida.
2. Um colega afirma: *"Python é uma linguagem orientada a objetos"*. Usando o mapa, formule uma
   resposta de três linhas que seja simultaneamente educada e tecnicamente precisa.
3. Qual região do mapa está **vazia** nesta modelagem? Isso é uma limitação da taxonomia, uma
   limitação da minha tabela, ou uma combinação genuinamente rara? Defenda sua resposta.
